# Image Classification with YOLOv8 (Ultralytics)

Fine-tune a YOLOv8 classification model on a custom Roboflow dataset, review the
results, validate the model, and run inference.

This notebook is organized into clear stages:

1. Setup (GPU check + install)
2. Download dataset from Roboflow
3. Train the classification model
4. Review training results
5. Validate the model
6. Run inference on test images
7. Save results to Google Drive

## 1. Setup

### Check GPU

Confirm a GPU is available. If not, go to `Runtime` -> `Change runtime type`
and set the hardware accelerator to `GPU`.

In [ ]:
!nvidia-smi

### Install dependencies

In [ ]:
%pip install ultralytics roboflow
import ultralytics
ultralytics.checks()

### Define a HOME constant

`HOME` makes it easy to reference datasets, runs, and weights with absolute paths.

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## 2. Download dataset from Roboflow

**Security note:** never hard-code your API key in the notebook. Store it in
Colab Secrets (left sidebar, key icon) under the name `ROBOFLOW_API_KEY`, then
load it as shown below.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('jfvorano').project('sampleproj-olscq')
version = project.version(2)
dataset = version.download('folder')
print('Dataset downloaded to:', dataset.location)

## 3. Train the classification model

**Important — check your dataset splits first.** For classification, every split
(`train`, `valid`, `test`) must contain a subfolder for *each* class. If `valid`
or `test` is missing a class, validation metrics will be meaningless. Aim for a
balanced dataset with enough images per class (tens to hundreds, not a handful).

Notes on the settings:
- `model=yolov8n-cls.pt` is the smallest YOLOv8 classification model.
- `imgsz=224` is the standard input size for classification (640 wastes compute).
- Increase `epochs` once you confirm the data and splits are correct.

In [ ]:
!yolo task=classify mode=train model=yolov8n-cls.pt data={dataset.location} \
      epochs=10 batch=32 imgsz=224 plots=True patience=40

## 4. Review training results

Classification results are saved under `runs/classify/train/`.

In [ ]:
!ls {HOME}/runs/classify/train/

### Confusion matrix

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/classify/train/confusion_matrix.png', width=600)

### Training curves

In [ ]:
IPyImage(filename=f'{HOME}/runs/classify/train/results.png', width=600)

### Sample validation predictions

In [ ]:
IPyImage(filename=f'{HOME}/runs/classify/train/val_batch0_pred.jpg', width=600)

## 5. Validate the fine-tuned model

Run validation against the best checkpoint. (For classification there is no
`data.yaml`; point `data` at the dataset folder.)

In [ ]:
!yolo task=classify mode=val \
      model={HOME}/runs/classify/train/weights/best.pt \
      data={dataset.location}

## 6. Run inference on test images

Load the trained model and predict the class of each test image, printing the
top class and its confidence.

In [ ]:
import os
from ultralytics import YOLO

model = YOLO(f'{HOME}/runs/classify/train/weights/best.pt')

test_dir = f'{dataset.location}/test'
results = model.predict(source=test_dir, imgsz=224, save=True)

for r in results:
    top_idx = int(r.probs.top1)
    top_conf = float(r.probs.top1conf)
    print(f'{os.path.basename(r.path)} -> {r.names[top_idx]} ({top_conf:.2f})')

## 7. Save results to Google Drive

Colab storage is temporary. Copy the `runs/` folder to Drive so your trained
weights and plots persist.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

source_folder = '/content/runs'
destination_folder = '/content/drive/MyDrive/CLASSIFICATION_TRAINING/'

os.makedirs(destination_folder, exist_ok=True)
shutil.copytree(source_folder, destination_folder, dirs_exist_ok=True)

print(f'Output successfully copied to {destination_folder}')

## Done

### Learning resources
- [Ultralytics Classification docs](https://docs.ultralytics.com/tasks/classify/)
- [Roboflow Notebooks](https://github.com/roboflow/notebooks)
- [Ultralytics train mode](https://docs.ultralytics.com/modes/train/)